# LLM-as-judge untuk IOTN-AC — multi-model, tiga kondisi, dengan pengukuran reliabilitas

Keluaran per foto: **segmentasi gigi + label FDI + landmark + grade AC 1–10 + persentase
keyakinan + catatan penalaran.**

## Yang sudah diketahui dari literatur

| Studi | Data | Hasil |
|---|---|---|
| [Diagnostics 2025, 15(23):3048](https://www.mdpi.com/2075-4418/15/23/3048) | 150 foto intraoral frontal, GPT-5 vs 2 klinisi (κ=0.91, ICC=0.88 antar klinisi) | **MAE 1.47**, akurasi klasifikasi **66.7%**, tanpa bias sistematis (Bland–Altman) |
| [Bioengineering 2024, 11(9):861](https://doi.org/10.3390/bioengineering11090861) | 1009 foto + overjet, CNN khusus | AC 1–10 **tidak tercapai**; hanya biner (82%) |

Jadi target realistis notebook ini: **MAE sekitar 1.5** dan akurasi pita di kisaran 60–70%.
Kalau hasilmu jauh di atas itu, curigai kebocoran atau bug — bukan terobosan.

Kesimpulan penulis Diagnostics layak dikutip apa adanya:
*"statistical accuracy alone is insufficient for safe clinical use."*

## Dua keputusan desain yang perlu kamu pahami

**1. Keyakinan diukur, bukan ditanyakan.** Keyakinan yang dilaporkan sendiri oleh LLM terkenal
tidak terkalibrasi — model bisa bilang "95% yakin" pada tebakan asal. Karena itu keyakinan utama
di sini dihitung dari **self-consistency**: setiap foto dinilai `k` kali secara independen pada
temperature > 0, lalu

```
grade      = median dari k sampel
confidence = proporsi sampel yang jatuh dalam ±1 dari median
```

Ini besaran empiris yang bisa diverifikasi. Keyakinan yang dilaporkan model tetap dicatat, tapi
sebagai kolom sekunder yang ditandai jelas.

**2. Tiga kondisi bertingkat.** Setiap foto dinilai dalam tiga kondisi:

| Kondisi | Yang dilihat model |
|---|---|
| **A** | Foto asli saja (setara studi Diagnostics — baseline yang sebanding) |
| **B** | Foto + overlay segmentasi, label FDI, dan landmark |
| **C** | Kondisi B + tabel metrik geometri ternormalisasi |

Membandingkan A/B/C menjawab pertanyaan yang belum ada di literatur: **apakah memberi model
struktur geometri eksplisit memperbaiki penilaiannya, atau justru menyesatkan?** Ini kontribusi
tersendiri yang layak masuk skripsi, terlepas dari arah hasilnya.

Sepuluh anchor AC resmi disertakan di **semua** kondisi — instrumen AC memang mensyaratkan
penilaian relatif terhadap foto rujukan, bukan penilaian absolut.

## 0. Model gratis dulu

Prioritas ke yang tidak berbayar. Tiga jalur, semuanya opsional — notebook melewati backend yang
kuncinya tidak tersedia.

| Backend | Biaya | Cara |
|---|---|---|
| **OpenRouter** | Gratis (model bersufiks `:free`) | Daftar, buat key. Tanpa kartu kredit. Batas ~20 req/menit, ~200 req/hari |
| **Google Gemini** | Ada free tier | `GEMINI_API_KEY` dari AI Studio |
| **Lokal** | Gratis penuh, offline | Qwen2.5-VL lewat `transformers`. Paling reprodusibel — bobotnya tetap. Berat di Mac; pakai varian 3B |
| Anthropic / OpenAI | Berbayar | Opsional, untuk pembanding |

```bash
pip install openai google-genai        # openai dipakai jg utk OpenRouter (API-nya kompatibel)
export OPENROUTER_API_KEY=sk-or-...
export GEMINI_API_KEY=...
```

**Catatan reproduktibilitas untuk skripsi:** model API bisa berubah diam-diam di balik nama yang
sama. Catat **tanggal, nama model persis, dan seed/temperature** di metodologi. Kalau reprodusibilitas
mutlak diperlukan, jalur lokal adalah satu-satunya yang bobotnya benar-benar tetap.

Batas kuota gratis (~200 req/hari) vs kebutuhan: `18 foto x 3 kondisi x k sampel x n model`.
Dengan k=5 dan 2 model itu **540 request** — melebihi kuota harian. Set `CFG["k_samples"]=3` dan
jalankan per model, atau pakai `CFG["subset"]` untuk uji coba dulu.

In [ ]:
# ========= CONFIG =========
CFG = {
    "img_dir":    "Front Teeth drg Laura",
    "seg_dir":    "seg_out",          # hasil notebook 3 (fdi.json, metrics_geometry.csv, masks.npz)
    "anchor_grid":"ac_reference_grid.png",
    "out_dir":    "judge_out",

    # model: (nama_pendek, backend, model_id). Backend tanpa API key otomatis dilewati.
    "models": [
        ("gemma4-free",  "openrouter", "google/gemma-4-31b-it:free"),
        ("nemotron-free","openrouter", "nvidia/nemotron-nano-12b-v2-vl:free"),
        ("gemini-flash", "gemini",     "gemini-2.5-flash"),
        # ("qwen-local", "local",      "Qwen/Qwen2.5-VL-3B-Instruct"),
        # ("claude",     "anthropic",  "claude-sonnet-5"),
        # ("gpt",        "openai",     "gpt-5"),
    ],

    "conditions": ["A", "B", "C"],    # A=foto, B=+overlay, C=+metrik
    "k_samples":  3,                  # sampel per (foto, kondisi, model) utk self-consistency
    "temperature": 0.7,               # >0 wajib, kalau 0 semua sampel identik & confidence palsu
    "subset":     None,               # mis. 4 = hanya 4 foto pertama (uji coba hemat kuota)
    "max_side":   900,                # resize gambar sebelum dikirim (hemat token)
    "retry":      3,
    "sleep":      3.5,                # jeda antar request (batas free tier ~20/menit)
}
import os, json
os.makedirs(CFG["out_dir"], exist_ok=True)
print("CFG:", json.dumps({k:v for k,v in CFG.items() if k!="models"}, indent=1))
print("model:", [m[0] for m in CFG["models"]])

## 1. Menyiapkan stimulus

Kondisi B membutuhkan overlay dari notebook 3. Kalau `seg_out/` belum ada, notebook ini tetap
jalan — kondisi B dan C otomatis dilewati dan hanya A yang dievaluasi.

In [ ]:
import glob, base64, time, re, io as _io
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
try:
    from pillow_heif import register_heif_opener; register_heif_opener()
except Exception: pass

EXTS = ("*.jpg","*.jpeg","*.JPG","*.JPEG","*.png","*.PNG","*.heic","*.HEIC")
paths = sorted(set(sum([glob.glob(os.path.join(CFG["img_dir"], e)) for e in EXTS], [])))
names = [os.path.splitext(os.path.basename(p))[0] for p in paths]
if CFG["subset"]:
    paths, names = paths[:CFG["subset"]], names[:CFG["subset"]]
print(f"{len(paths)} foto")

def load_img(p, max_side=None):
    im = Image.open(p).convert("RGB")
    ms = max_side or CFG["max_side"]
    if max(im.size) > ms:
        sc = ms/max(im.size); im = im.resize((int(im.width*sc), int(im.height*sc)), Image.LANCZOS)
    return im

# --- anchor AC: dikirim di SEMUA kondisi ---
ANCHOR = load_img(CFG["anchor_grid"], 1100) if os.path.exists(CFG["anchor_grid"]) else None
print("anchor grid:", "ada" if ANCHOR else "TIDAK ADA -> penilaian jadi absolut, kurang valid")

# --- hasil notebook 3 ---
SEG_OK = os.path.exists(os.path.join(CFG["seg_dir"], "fdi.json"))
MET = None
if SEG_OK:
    MET = pd.read_csv(os.path.join(CFG["seg_dir"], "metrics_geometry.csv"))
    MET = MET.rename(columns={MET.columns[0]: "foto"}).set_index("foto")
    with open(os.path.join(CFG["seg_dir"], "fdi.json")) as f: FDI = json.load(f)
    MASKS = np.load(os.path.join(CFG["seg_dir"], "masks.npz")) if \
            os.path.exists(os.path.join(CFG["seg_dir"], "masks.npz")) else None
    print("seg_out ditemukan — kondisi B & C aktif")
else:
    print("seg_out TIDAK ada — hanya kondisi A yang dijalankan")
    CFG["conditions"] = ["A"]

In [ ]:
def make_overlay(name, p):
    """Foto + kontur mask + nomor FDI. Mengembalikan None kalau data seg tak lengkap."""
    if not SEG_OK or MASKS is None or name not in FDI or not FDI[name]:
        return None
    im = load_img(p, 1024)
    W0, H0 = im.size
    fig = plt.figure(figsize=(W0/100, H0/100), dpi=100)
    ax = fig.add_axes([0,0,1,1]); ax.imshow(im); ax.axis("off")
    drew = 0
    for k, fdi_num in FDI[name].items():
        key = f"{name}__{k}"
        if key not in MASKS.files: continue
        m = MASKS[key]
        if m.shape != (H0, W0):
            m = np.asarray(Image.fromarray(m.astype(np.uint8)*255).resize((W0,H0),
                           Image.NEAREST), bool)
        ys, xs = np.nonzero(m)
        if len(xs) < 20: continue
        up = int(fdi_num)//10 in (1,2)
        col = "#00c000" if up else "#ff3030"
        ax.contour(m, levels=[0.5], colors=col, linewidths=1.6)
        ax.text(xs.mean(), ys.mean(), str(fdi_num), color="white", fontsize=11, weight="bold",
                ha="center", va="center",
                bbox=dict(fc=col, ec="none", alpha=0.85, pad=1.2))
        ax.plot(xs.mean(), ys.max() if up else ys.min(), "^", ms=6, color="#1f77b4")
        drew += 1
    buf = _io.BytesIO(); fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    buf.seek(0)
    return load_img_bytes(buf) if drew else None

def load_img_bytes(buf):
    im = Image.open(buf).convert("RGB")
    if max(im.size) > CFG["max_side"]:
        sc = CFG["max_side"]/max(im.size)
        im = im.resize((int(im.width*sc), int(im.height*sc)), Image.LANCZOS)
    return im

def metrics_text(name):
    if MET is None or name not in MET.index: return None
    r = MET.loc[name]
    if r.get("valid") is not True and str(r.get("valid")).lower() != "true": return None
    keys = ["LII_norm","LII_max","incisal_rms","tilt_std","tilt_asym","width_asym",
            "solidity_min","midline_dev","cant_deg","wh_central","overbite_proxy"]
    out = []
    for k in keys:
        v = pd.to_numeric(pd.Series([r.get(k)]), errors="coerce").iloc[0]
        if pd.notna(v): out.append(f"  {k} = {v:.3f}")
    susp = " (PERINGATAN: penomoran FDI meragukan)" if r.get("fdi_suspect") in (True,"True") else ""
    return f"Metrik geometri ternormalisasi (dibagi lebar insisivus sentral){susp}:\n" + "\n".join(out)

STIM = {}
for p, n in zip(paths, names):
    STIM[n] = {"A": load_img(p), "B": make_overlay(n, p), "C_text": metrics_text(n)}
okB = sum(1 for n in STIM if STIM[n]["B"] is not None)
okC = sum(1 for n in STIM if STIM[n]["C_text"] is not None)
print(f"overlay siap: {okB}/{len(STIM)} | tabel metrik siap: {okC}/{len(STIM)}")

In [ ]:
# Intip satu overlay untuk memastikan yang dilihat model memang masuk akal
_n = next((n for n in STIM if STIM[n]["B"] is not None), None)
if _n:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    ax[0].imshow(STIM[_n]["A"]); ax[0].set_title(f"Kondisi A — {_n[:34]}", fontsize=9)
    ax[1].imshow(STIM[_n]["B"]); ax[1].set_title("Kondisi B — overlay", fontsize=9)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    print("Kondisi C menambahkan teks ini:\n"); print(STIM[_n]["C_text"])
else:
    print("Belum ada overlay — jalankan notebook 3 dulu kalau ingin kondisi B & C.")

## 2. Prompt

Dirancang mengikuti cara instrumen AC sebenarnya dipakai: **bandingkan dengan 10 foto rujukan**,
bukan menilai dari nol. Model juga diminta menyebut kriteria yang ia amati **sebelum** memberi
angka — urutan ini mengurangi kecenderungan menebak gestalt lalu mengarang alasan.

Keluaran dipaksa JSON supaya bisa di-parse. Field `confidence_reported` sengaja diminta tapi
**tidak** dipakai sebagai keyakinan utama.

In [ ]:
SYSTEM = (
 "Anda adalah asisten yang menilai estetika gigi anterior memakai Aesthetic Component (AC) "
 "dari Index of Orthodontic Treatment Need. Anda teliti, kalibrasi Anda konservatif, dan Anda "
 "menyatakan ketidakpastian secara jujur. Jawab HANYA dengan satu objek JSON valid."
)

RUBRIC = """
Skala AC adalah 1-10 berdasarkan DAYA TARIK gigi secara keseluruhan:
  1  = paling menarik: gigi sejajar rapi, lengkung teratur, tanpa celah/berjejal berarti
  10 = paling tidak menarik: berjejal/rotasi/celah berat, gigi jauh keluar lengkung

Pita kebutuhan perawatan (Richmond dkk.):
  1-4  = tidak perlu perawatan
  5-7  = borderline
  8-10 = jelas perlu perawatan

Gambar pertama adalah KARTU RUJUKAN RESMI berisi 10 foto anchor bernomor 1-10.
Nilai foto pasien dengan MEMBANDINGKANNYA ke anchor tersebut, bukan menilai absolut.
Penilaian adalah daya tarik keseluruhan, bukan kemiripan detail dengan satu anchor.

Pertimbangkan: keberjejalan/iregularitas, celah/diastema, rotasi, pergeseran midline,
kesejajaran tepi insisal, simetri kiri-kanan, gigi hilang/terhalang, keterlihatan gigi bawah.
JANGAN menilai warna gigi atau kondisi gusi — AC menilai posisi dan susunan.
"""

TASK = """
Kerjakan berurutan:
1. Sebutkan ciri yang Anda AMATI (isi "observations"), sebelum menentukan angka.
2. Tentukan anchor mana yang paling mendekati ("closest_anchor").
3. Baru berikan "ac_grade" (bilangan bulat 1-10) dan "band".
4. Sebutkan hal yang membuat Anda ragu di "uncertainty".

Balas HANYA JSON dengan struktur persis ini:
{
  "observations": {
    "crowding": "<ringkas>",
    "spacing": "<ringkas>",
    "rotation": "<ringkas>",
    "midline": "<ringkas>",
    "incisal_alignment": "<ringkas>",
    "symmetry": "<ringkas>"
  },
  "closest_anchor": <1-10>,
  "ac_grade": <1-10>,
  "band": "<1-4|5-7|8-10>",
  "confidence_reported": <0-100>,
  "reasoning": "<2-4 kalimat alasan grade tsb>",
  "uncertainty": "<apa yang membatasi penilaian Anda>"
}
"""

COND_NOTE = {
 "A": "",
 "B": ("\nGambar terakhir adalah foto yang sama dengan overlay hasil segmentasi otomatis: "
       "kontur tiap gigi, nomor FDI (hijau=rahang atas, merah=rahang bawah), dan segitiga biru "
       "menandai tepi insisal. Overlay ini dihasilkan otomatis dan BISA SALAH — pakai sebagai "
       "bantuan, bukan kebenaran. Kalau bertentangan dengan yang Anda lihat, percayai foto."),
 "C": ("\nGambar terakhir adalah foto dengan overlay segmentasi (kontur gigi, nomor FDI, tepi "
       "insisal). Disertakan pula metrik geometri terukur. Semua metrik TANPA SATUAN karena "
       "dibagi lebar insisivus sentral. Nilai lebih tinggi = lebih tidak teratur, kecuali "
       "solidity_min (lebih rendah = siluet lebih terpotong). Overlay dan metrik dihasilkan "
       "otomatis dan BISA SALAH."),
}

def build_prompt(name, cond):
    txt = RUBRIC + COND_NOTE[cond]
    if cond == "C" and STIM[name]["C_text"]:
        txt += "\n\n" + STIM[name]["C_text"]
    imgs = ([ANCHOR] if ANCHOR else []) + [STIM[name]["A"]]
    if cond in ("B","C") and STIM[name]["B"] is not None:
        imgs.append(STIM[name]["B"])
    return txt + "\n" + TASK, imgs

def available_conditions(name):
    out = ["A"]
    if "B" in CFG["conditions"] and STIM[name]["B"] is not None: out.append("B")
    if "C" in CFG["conditions"] and STIM[name]["B"] is not None and STIM[name]["C_text"]: out.append("C")
    return [c for c in out if c in CFG["conditions"]]

_t, _i = build_prompt(names[0], "A")
print(f"prompt kondisi A: {len(_t)} karakter, {len(_i)} gambar")

## 3. Backend model — antarmuka tunggal

Semua backend dibungkus jadi satu fungsi `call(model, prompt, images) -> str`. Backend yang
API key-nya tidak ada otomatis dilewati, jadi notebook tetap jalan dengan model apa pun yang
kamu punya.

In [ ]:
def b64(im, fmt="JPEG"):
    buf = _io.BytesIO(); im.save(buf, fmt, quality=88)
    return base64.b64encode(buf.getvalue()).decode()

def _openai_style(model_id, prompt, images, base_url, api_key, temp):
    from openai import OpenAI
    cli = OpenAI(base_url=base_url, api_key=api_key)
    content = [{"type":"text","text":prompt}]
    for im in images:
        content.append({"type":"image_url",
                        "image_url":{"url":f"data:image/jpeg;base64,{b64(im)}"}})
    r = cli.chat.completions.create(
        model=model_id, temperature=temp, max_tokens=1200,
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":content}])
    return r.choices[0].message.content

def _gemini(model_id, prompt, images, temp):
    from google import genai
    from google.genai import types
    cli = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    parts = [types.Part.from_text(text=SYSTEM + "\n\n" + prompt)]
    for im in images:
        parts.append(types.Part.from_bytes(
            data=base64.b64decode(b64(im)), mime_type="image/jpeg"))
    r = cli.models.generate_content(
        model=model_id, contents=parts,
        config=types.GenerateContentConfig(temperature=temp, max_output_tokens=1200))
    return r.text

def _anthropic(model_id, prompt, images, temp):
    import anthropic
    cli = anthropic.Anthropic()
    content = []
    for im in images:
        content.append({"type":"image","source":{"type":"base64",
                        "media_type":"image/jpeg","data":b64(im)}})
    content.append({"type":"text","text":prompt})
    r = cli.messages.create(model=model_id, max_tokens=1200, temperature=temp,
                            system=SYSTEM, messages=[{"role":"user","content":content}])
    return r.content[0].text

_LOCAL = {}
def _local(model_id, prompt, images, temp):
    import torch
    from transformers import AutoProcessor, AutoModelForImageTextToText
    if model_id not in _LOCAL:
        pr = AutoProcessor.from_pretrained(model_id)
        mo = AutoModelForImageTextToText.from_pretrained(
            model_id, dtype="auto",
            device_map="mps" if torch.backends.mps.is_available() else "auto")
        _LOCAL[model_id] = (pr, mo)
    pr, mo = _LOCAL[model_id]
    msg = [{"role":"user","content":[{"type":"image"} for _ in images] +
                                    [{"type":"text","text":SYSTEM+"\n\n"+prompt}]}]
    text = pr.apply_chat_template(msg, add_generation_prompt=True)
    inp = pr(text=[text], images=images, return_tensors="pt").to(mo.device)
    out = mo.generate(**inp, max_new_tokens=900, do_sample=temp > 0, temperature=max(temp,1e-3))
    return pr.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def call(backend, model_id, prompt, images, temp):
    if backend == "openrouter":
        return _openai_style(model_id, prompt, images,
                             "https://openrouter.ai/api/v1", os.environ["OPENROUTER_API_KEY"], temp)
    if backend == "openai":
        return _openai_style(model_id, prompt, images, None, os.environ["OPENAI_API_KEY"], temp)
    if backend == "gemini":    return _gemini(model_id, prompt, images, temp)
    if backend == "anthropic": return _anthropic(model_id, prompt, images, temp)
    if backend == "local":     return _local(model_id, prompt, images, temp)
    raise ValueError(backend)

NEEDS_KEY = {"openrouter":"OPENROUTER_API_KEY", "openai":"OPENAI_API_KEY",
             "gemini":"GEMINI_API_KEY", "anthropic":"ANTHROPIC_API_KEY", "local":None}
ACTIVE = []
for short, backend, mid in CFG["models"]:
    key = NEEDS_KEY[backend]
    if key and not os.environ.get(key):
        print(f"  lewati {short:15s} ({backend}) — {key} tidak diset"); continue
    ACTIVE.append((short, backend, mid)); print(f"  aktif  {short:15s} {backend:11s} {mid}")
if not ACTIVE:
    print("\nTidak ada model aktif. Set minimal satu API key, atau aktifkan backend 'local' di CFG.")

## 4. Menjalankan penilaian

Estimasi request: `foto x kondisi x k_samples x model`. Dengan 18 foto, 3 kondisi, k=3, 2 model
= **324 request**, di atas kuota harian free tier (~200). Pakai `CFG["subset"]` untuk uji dulu,
atau jalankan per model di hari berbeda.

Hasil ditulis inkremental ke `judge_out/raw_responses.jsonl`, jadi kalau terputus tinggal
jalankan ulang — yang sudah selesai dilewati.

In [ ]:
RAW_PATH = os.path.join(CFG["out_dir"], "raw_responses.jsonl")

def parse_json(txt):
    if not txt: return None
    m = re.search(r"\{.*\}", txt, re.S)
    if not m: return None
    s = m.group(0)
    for attempt in (s, s.replace("\n", " "), re.sub(r",\s*([}\]])", r"\1", s)):
        try: return json.loads(attempt)
        except Exception: pass
    return None

done = set()
if os.path.exists(RAW_PATH):
    for line in open(RAW_PATH):
        try:
            d = json.loads(line); done.add((d["model"], d["foto"], d["cond"], d["sample"]))
        except Exception: pass
print(f"{len(done)} penilaian sudah ada di cache")

jobs = [(s, b, mid, n, c, k)
        for (s, b, mid) in ACTIVE for n in names
        for c in available_conditions(n) for k in range(CFG["k_samples"])
        if (s, n, c, k) not in done]
print(f"{len(jobs)} request akan dijalankan\n")

fh = open(RAW_PATH, "a")
t0 = time.time()
for i, (short, backend, mid, name, cond, k) in enumerate(jobs):
    prompt, images = build_prompt(name, cond)
    txt, err = None, None
    for attempt in range(CFG["retry"]):
        try:
            txt = call(backend, mid, prompt, images, CFG["temperature"]); break
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"
            time.sleep(CFG["sleep"]*(attempt+1))
    rec = {"model": short, "backend": backend, "model_id": mid, "foto": name, "cond": cond,
           "sample": k, "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
           "temperature": CFG["temperature"], "raw": txt, "error": err,
           "parsed": parse_json(txt)}
    fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()
    g = (rec["parsed"] or {}).get("ac_grade", "?")
    if (i+1) % 5 == 0 or i == 0:
        el = time.time()-t0
        print(f"  [{i+1}/{len(jobs)}] {short} {name[:26]:26s} {cond} s{k} -> AC {g}"
              f"   ({el/max(i+1,1):.1f}s/req, sisa ~{el/max(i+1,1)*(len(jobs)-i-1)/60:.0f} mnt)")
    time.sleep(CFG["sleep"])
fh.close()
print(f"\nselesai dalam {(time.time()-t0)/60:.1f} menit -> {RAW_PATH}")

## 5. Agregasi & keyakinan terukur

```
grade      = median dari k sampel
confidence = proporsi sampel dalam ±1 dari median   (0-100%)
spread     = simpangan baku antar sampel
```

`confidence` inilah keyakinan yang bisa dipertanggungjawabkan. `confidence_reported` dari model
ikut dicatat supaya kamu bisa **menguji kalibrasinya** — kalau korelasi keduanya rendah,
itu bukti langsung bahwa keyakinan yang dilaporkan model tidak bermakna.

In [ ]:
recs = [json.loads(l) for l in open(RAW_PATH)] if os.path.exists(RAW_PATH) else []
rows = []
for r in recs:
    p = r.get("parsed") or {}
    g = p.get("ac_grade")
    try: g = int(g)
    except Exception: g = np.nan
    rows.append({"model": r["model"], "foto": r["foto"], "cond": r["cond"], "sample": r["sample"],
                 "grade": g, "band": p.get("band"), "anchor": p.get("closest_anchor"),
                 "conf_rep": p.get("confidence_reported"),
                 "reasoning": p.get("reasoning"), "uncertainty": p.get("uncertainty"),
                 "obs": json.dumps(p.get("observations", {}), ensure_ascii=False),
                 "ok": bool(p) and not np.isnan(g)})
R = pd.DataFrame(rows)
if len(R):
    print(f"{len(R)} respons | ter-parse {R.ok.mean()*100:.0f}%")
    bad = R[~R.ok]
    if len(bad): print("gagal parse per model:\n", bad.groupby("model").size().to_string())

def agg(g):
    v = g.grade.dropna().values
    if len(v) == 0: return pd.Series({"ac": np.nan, "confidence": np.nan, "spread": np.nan, "n": 0})
    med = float(np.median(v))
    return pd.Series({"ac": med,
                      "confidence": 100.0*np.mean(np.abs(v-med) <= 1),
                      "spread": float(np.std(v)),
                      "conf_rep": float(pd.to_numeric(g.conf_rep, errors="coerce").mean()),
                      "n": len(v)})

A = R.groupby(["model","cond","foto"]).apply(agg, include_groups=False).reset_index() if len(R) else pd.DataFrame()
if len(A):
    A.to_csv(os.path.join(CFG["out_dir"], "ac_aggregated.csv"), index=False)
    print("\n", A.groupby(["model","cond"])[["ac","confidence","spread"]].mean().round(2).to_string())

## 6. Analisis

Tiga pertanyaan:

1. **Apakah kondisi B/C mengalahkan A?** — apakah overlay & metrik membantu
2. **Seberapa sepakat antar model?** — ICC & weighted kappa
3. **Apakah `confidence_reported` terkalibrasi?** — korelasinya dengan self-consistency

In [ ]:
from scipy.stats import spearmanr
from itertools import combinations

if len(A):
    print("=== 1. Perbandingan kondisi (per model) ===")
    for m in A.model.unique():
        sub = A[A.model == m]
        piv = sub.pivot(index="foto", columns="cond", values="ac")
        line = f"  {m:15s}"
        for c in ["A","B","C"]:
            if c in piv: line += f"  {c}: mean={piv[c].mean():.2f} sd={piv[c].std():.2f}"
        for c in ["B","C"]:
            if c in piv and "A" in piv:
                d = piv[[c,"A"]].dropna()
                if len(d) > 3:
                    line += f"  | rho({c},A)={spearmanr(d[c], d['A'])[0]:+.2f}"
        print(line)

    print("\n=== 2. Kesepakatan antar model (kondisi A) ===")
    pa = A[A.cond == "A"].pivot(index="foto", columns="model", values="ac").dropna()
    if pa.shape[1] >= 2:
        for m1, m2 in combinations(pa.columns, 2):
            rho = spearmanr(pa[m1], pa[m2])[0]
            mae = np.abs(pa[m1]-pa[m2]).mean()
            print(f"  {m1} vs {m2}: rho={rho:+.3f}  MAE={mae:.2f}")
        # ICC(2,1) two-way random, absolute agreement
        Xm = pa.values; n, k = Xm.shape
        gm = Xm.mean()
        MSR = k*((Xm.mean(1)-gm)**2).sum()/(n-1)
        MSC = n*((Xm.mean(0)-gm)**2).sum()/(k-1)
        MSE = ((Xm - Xm.mean(1,keepdims=True) - Xm.mean(0,keepdims=True) + gm)**2).sum()/((n-1)*(k-1))
        icc = (MSR-MSE)/(MSR+(k-1)*MSE+k*(MSC-MSE)/n)
        print(f"  ICC(2,1) = {icc:.3f}   (<0.5 buruk | 0.5-0.75 sedang | 0.75-0.9 baik | >0.9 sangat baik)")
    else:
        print("  perlu >=2 model untuk uji kesepakatan")

    print("\n=== 3. Kalibrasi confidence yang dilaporkan model ===")
    d = A.dropna(subset=["confidence","conf_rep"])
    if len(d) > 5:
        rho, p = spearmanr(d.conf_rep, d.confidence)
        print(f"  rho(confidence_reported, self-consistency) = {rho:+.3f} (p={p:.3f})")
        print("  -> mendekati 0 berarti angka keyakinan dari model TIDAK bermakna;")
        print("     pakai kolom 'confidence' hasil self-consistency.")

In [ ]:
# Bandingkan dengan label sementara & metrik geometri, kalau ada
ext = "labels/ac_llm_provisional.csv"
if len(A) and os.path.exists(ext):
    P = pd.read_csv(ext)[["foto","ac_llm"]]
    for m in A.model.unique():
        for c in sorted(A[A.model == m].cond.unique()):
            d = A[(A.model == m) & (A.cond == c)].merge(P, on="foto").dropna(subset=["ac","ac_llm"])
            if len(d) > 4:
                rho = spearmanr(d.ac, d.ac_llm)[0]
                print(f"  {m:15s} {c}: vs label sementara  rho={rho:+.3f}  MAE={np.abs(d.ac-d.ac_llm).mean():.2f}  n={len(d)}")

if len(A) and MET is not None:
    print()
    num = MET.apply(pd.to_numeric, errors="coerce")
    for m in A.model.unique():
        d = A[(A.model == m) & (A.cond == "A")].set_index("foto").join(num, how="inner")
        for col in ["LII_norm","incisal_rms","midline_dev"]:
            if col in d and d[col].notna().sum() > 4:
                rho = spearmanr(d.ac, d[col], nan_policy="omit")[0]
                print(f"  {m:15s} A: vs {col:12s} rho={rho:+.3f} n={d[col].notna().sum()}")

## 7. Kartu laporan per foto

Satu PNG per foto berisi persis yang kamu minta: **foto asli, overlay segmentasi + FDI +
landmark, grade AC, persentase keyakinan, dan catatan penalaran** dari tiap model.

Ini juga format yang enak dibawa ke drg. Laura — beliau bisa langsung menunjuk mana yang
tidak setuju, dan alasan model tertulis di sana untuk diperdebatkan.

In [ ]:
import textwrap
def report_card(name, save=True):
    sub = A[A.foto == name] if len(A) else pd.DataFrame()
    fig = plt.figure(figsize=(13, 7.6))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.25, 1], hspace=0.18, wspace=0.06)

    ax1 = fig.add_subplot(gs[0,0]); ax1.imshow(STIM[name]["A"]); ax1.axis("off")
    ax1.set_title("Foto asli", fontsize=10)
    ax2 = fig.add_subplot(gs[0,1]); ax2.axis("off")
    if STIM[name]["B"] is not None:
        ax2.imshow(STIM[name]["B"]); ax2.set_title("Segmentasi + FDI + landmark", fontsize=10)
    else:
        ax2.text(.5,.5,"segmentasi tidak tersedia",ha="center",va="center",color="gray")

    ax3 = fig.add_subplot(gs[1,:]); ax3.axis("off")
    y = 0.97
    ax3.text(0, y, name, fontsize=11, weight="bold", va="top"); y -= 0.12
    if len(sub):
        for _, r in sub.sort_values(["model","cond"]).iterrows():
            if np.isnan(r.ac): continue
            txt = (f"{r.model} [{r.cond}]  →  AC {r.ac:.0f}   "
                   f"keyakinan {r.confidence:.0f}%   (sebaran {r.spread:.2f}, n={int(r['n'])})")
            ax3.text(0, y, txt, fontsize=9.5, weight="bold", va="top",
                     color={"A":"#1f77b4","B":"#2ca02c","C":"#d62728"}.get(r.cond,"k"))
            y -= 0.10
            rs = R[(R.foto==name)&(R.model==r.model)&(R.cond==r.cond)&R.ok]
            reason = rs.reasoning.dropna().iloc[0] if len(rs.reasoning.dropna()) else ""
            for ln in textwrap.wrap(str(reason), 150)[:3]:
                ax3.text(0.02, y, ln, fontsize=8.4, va="top", color="#333"); y -= 0.085
            y -= 0.03
    else:
        ax3.text(0, y, "belum ada penilaian", fontsize=10, color="gray", va="top")
    if save:
        f = os.path.join(CFG["out_dir"], f"card_{re.sub(r'[^A-Za-z0-9]+','_',name)[:50]}.png")
        fig.savefig(f, dpi=110, bbox_inches="tight")
    return fig

for n in names[:3]:
    report_card(n); plt.show()
print(f"\nKartu tersimpan di {CFG['out_dir']}/card_*.png")

In [ ]:
# Ekspor gabungan: satu baris per foto, siap dibawa ke dokter / dilampirkan di skripsi
if len(A):
    best = A[A.cond == ("C" if "C" in A.cond.unique() else A.cond.unique()[0])]
    piv = A.pivot_table(index="foto", columns=["model","cond"], values=["ac","confidence"])
    piv.columns = ["_".join(map(str, c)) for c in piv.columns]
    firstreason = (R[R.ok].sort_values(["foto","model","cond","sample"])
                     .groupby("foto").reasoning.first())
    OUT = piv.join(firstreason)
    OUT.to_csv(os.path.join(CFG["out_dir"], "ac_report.csv"))
    print(OUT.round(2).to_string()[:3000])
    print("\n->", os.path.join(CFG["out_dir"], "ac_report.csv"))

## Batas yang harus ikut tertulis di metodologi

- **Tidak ada LLM yang tervalidasi untuk IOTN-AC.** Patokan terbaik saat ini MAE ≈ 1.47
  (GPT-5, n=150). Perlakukan keluaran ini sebagai **label sementara**, bukan ground truth.
- **Model API bisa berubah di balik nama yang sama.** Catat tanggal, `model_id` persis, dan
  temperature. Jalur lokal satu-satunya yang bobotnya benar-benar tetap.
- **Keyakinan yang dilaporkan model tidak terkalibrasi.** Sel analisis mengujinya secara langsung.
- **Self-consistency mengukur kestabilan, bukan kebenaran.** Model bisa konsisten dan konsisten
  salah. Hanya perbandingan dengan penilaian dokter yang bisa menjawab kebenaran.
- **Kondisi B/C memakai overlay yang bisa salah.** Pada run terakhir hanya 9/18 foto valid dan
  5 di antaranya penomorannya meragukan — perbaiki segmentasi dulu sebelum menarik kesimpulan
  tentang apakah B/C lebih baik dari A.